# SignBridge — Notebook 1: Dataset Preparation & Signer Splits

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshiaradhya/SignBridge/blob/main/notebooks/01_dataset_preparation.ipynb)

This notebook handles:
1. Setting up the Google Colab environment and Google Drive persistence.
2. Downloading the Indian Sign Language (**INCLUDE / INCLUDE-50**) and American Sign Language (**ASL Citizen**) datasets.
3. Generating **signer-independent** train, validation, and test split manifests to prevent data leakage across signers.

### Dataset Licensing Notice
* **INCLUDE**: Created by AI4Bharat (IIT Madras). Released under Creative Commons Attribution 4.0 International (CC BY 4.0).
* **ASL Citizen**: Created by Microsoft Research and Deaf accessibility researchers (Desai et al.). Released for Academic Research Use.
SignBridge respects all licensing restrictions and preserves official video labels.

In [ ]:
# Step 1: Clone repository and install dependencies
!git clone https://github.com/joshiaradhya/SignBridge.git /content/SignBridge || (cd /content/SignBridge && git pull)
%cd /content/SignBridge
!pip install -q -r ml/colab-requirements.txt

In [ ]:
# Step 2: (Optional) Mount Google Drive for persistent dataset storage
from google.colab import drive
import os

MOUNT_DRIVE = True
if MOUNT_DRIVE:
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/SignBridge_Data'
    os.makedirs(data_dir, exist_ok=True)
    print(f'Persistent storage active at: {data_dir}')
else:
    data_dir = '/content/data'
    os.makedirs(data_dir, exist_ok=True)

In [ ]:
# Step 3: Prepare INCLUDE-50 dataset
# Place INCLUDE-50 archive in your Google Drive or download directly:
# Example layout: data/include50/<GLOSS>/<video_files>.mp4
import sys
sys.path.append('/content/SignBridge')

from pathlib import Path
from ml.signbridge.datasets import prepare_include_manifest, verify_signer_isolation

include_dir = Path(data_dir) / 'include50'
include_dir.mkdir(parents=True, exist_ok=True)

print('INCLUDE directory:', include_dir)
# If you have a zip file in Drive:
# !unzip -q -n /content/drive/MyDrive/include50.zip -d {include_dir}

# Build signer-independent manifest for INCLUDE-50
try:
    include_manifest = prepare_include_manifest(include_dir, val_ratio=0.15, test_ratio=0.15)
    valid, msg = verify_signer_isolation(include_manifest)
    print(msg)
    manifest_path = Path('data/manifests/include50_manifest.json')
    include_manifest.save(manifest_path)
    print(f'Wrote INCLUDE manifest to {manifest_path}')
except Exception as e:
    print('Note: Populate INCLUDE videos in directory before running manifest builder.', e)

In [ ]:
# Step 4: Prepare ASL Citizen dataset
from ml.signbridge.datasets import prepare_asl_citizen_manifest

asl_dir = Path(data_dir) / 'asl_citizen'
asl_csv = asl_dir / 'asl_citizen_metadata.csv'
asl_videos = asl_dir / 'videos'

try:
    asl_manifest = prepare_asl_citizen_manifest(asl_csv, asl_videos, val_ratio=0.15, test_ratio=0.15)
    valid, msg = verify_signer_isolation(asl_manifest)
    print(msg)
    asl_manifest_path = Path('data/manifests/asl_citizen_manifest.json')
    asl_manifest.save(asl_manifest_path)
    print(f'Wrote ASL Citizen manifest to {asl_manifest_path}')
except Exception as e:
    print('Note: Download ASL Citizen metadata CSV and video folder before building manifest.', e)